# Download `humair025/suno-audio`

This notebook downloads the dataset from Hugging Face using `snapshot_download(...)`.

Why this approach:
- it avoids the earlier audio-decoding error from `load_dataset(...)`,
- it is resumable,
- it works for selected batches or the whole repository,
- it can use your `HF_TOKEN` for authenticated requests.

Notes:
- the dataset is public, so a token is optional,
- a token mainly helps with authentication and higher rate limits,
- the full repository is very large, so this notebook defaults to downloading only `batch_0`,
- set `BATCHES_TO_DOWNLOAD = None` only if you intentionally want the full dataset.

In [ ]:
%pip install -q -U "huggingface_hub>=0.32.0" "hf-xet>=1.1.0" "datasets>=2.19.0"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

REPO_ID = "humair025/suno-audio"
DOWNLOAD_DIR = Path.cwd() / "data" / "suno-audio"

# Examples: [0], [0, 1, 2], list(range(5)), or None for the entire dataset.
BATCHES_TO_DOWNLOAD = [0]

MAX_WORKERS = 8
PROMPT_FOR_TOKEN = False
ENABLE_HIGH_PERFORMANCE = True

hf_token = os.getenv("HF_TOKEN")
if PROMPT_FOR_TOKEN and not hf_token:
    entered = getpass("HF token (optional, press Enter to skip): ").strip()
    hf_token = entered or None

if ENABLE_HIGH_PERFORMANCE:
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo: {REPO_ID}")
print(f"Download directory: {DOWNLOAD_DIR}")
print("Batches:", "all" if BATCHES_TO_DOWNLOAD is None else BATCHES_TO_DOWNLOAD)
print("Using token:", bool(hf_token))

In [ ]:
from huggingface_hub import snapshot_download


def build_allow_patterns(batches):
    if batches is None:
        return None

    normalized = sorted({int(batch) for batch in batches})
    return ["README.md", *[f"batch_{batch}/*" for batch in normalized]]


def format_bytes(num_bytes):
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:.2f} {unit}"
        value /= 1024


allow_patterns = build_allow_patterns(BATCHES_TO_DOWNLOAD)
allow_patterns

## Optional dry-run

This checks which files would be downloaded and estimates the size before the real transfer starts.

In [ ]:
dry_run_files = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=str(DOWNLOAD_DIR),
    token=hf_token,
    allow_patterns=allow_patterns,
    max_workers=MAX_WORKERS,
    dry_run=True,
)

bytes_to_download = sum(file.file_size for file in dry_run_files if file.will_download)
print(f"Files to download: {sum(file.will_download for file in dry_run_files)} / {len(dry_run_files)}")
print(f"Estimated bytes to download: {format_bytes(bytes_to_download)}")

for file in dry_run_files[:15]:
    status = "download" if file.will_download else "cached"
    print(f"[{status}] {file.filename} ({format_bytes(file.file_size)})")

if len(dry_run_files) > 15:
    print("...")

## Download

This is safe to rerun. Hugging Face will reuse existing local files and metadata when possible.

In [ ]:
download_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=str(DOWNLOAD_DIR),
    token=hf_token,
    allow_patterns=allow_patterns,
    max_workers=MAX_WORKERS,
)

print(f"Download complete: {download_path}")

## Optional verification

Load one downloaded batch from disk and inspect metadata without decoding audio.

In [ ]:
from datasets import load_from_disk

PREVIEW_BATCH = 0
batch_dir = DOWNLOAD_DIR / f"batch_{PREVIEW_BATCH}"

if not batch_dir.exists():
    raise FileNotFoundError(
        f"{batch_dir} does not exist yet. Download that batch first or change PREVIEW_BATCH."
    )

batch_ds = load_from_disk(str(batch_dir))
metadata_columns = [column for column in batch_ds.column_names if column != "audio"]
metadata_only = batch_ds.select_columns(metadata_columns)

print(batch_ds)
display(metadata_only.select(range(min(5, len(metadata_only)))).to_pandas())